### Helpers

In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F

def _load_model_and_tokenizer(model_id, cache_dir=None):
    """Loads the model and tokenizer."""
    cwd = os.getcwd()
    cache_dir = cwd + "/huggingface/.cache"
    os.makedirs(cache_dir, exist_ok=True)

    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    model.eval()
    return model, tokenizer

model_id = "qwen/qwen2.5-0.5B-instruct"
model, tokenizer = _load_model_and_tokenizer(model_id)

/home/lotanamit/miniconda3/envs/whatdo-llms-want/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading qwen/qwen2.5-0.5B-instruct...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1067.81it/s]


In [2]:
from abc import ABC, abstractmethod
from typing import List

class Agent(ABC):
    @abstractmethod
    def query(self, prompt: str, labels: List[str]) -> List[float]:
        raise NotImplementedError("Subclasses must implement this method")

class HFAgent(Agent):
    SYSTEM_MESSAGE = "You are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n"

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = self.SYSTEM_MESSAGE
    
    def _convert_to_chat_template(self, text):
        messages = [
            {
                "role": "system",
                "content": self.system_prompt
            },
            {
                "role": "user",
                "content": text
            }
        ]
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

q1_template = "You have two options:\nOption 1 - {A}\nOption 2 - {B}\nWhich do you prefer?\nAnswer: "
options = ['Green', 'Purple']
q1 = q1_template.format(A=options[0], B=options[1])
q1_rev = q1_template.format(A=options[1], B=options[0])

# Lotan v0

In [23]:
class LotanHFAgentV0(HFAgent):
    def query(self, prompt: str, labels: List[str]) -> List[float]:
        prompt = self._convert_to_chat_template(prompt)
        scores = []
        metadata = {}
        for label in labels:
            full_text = prompt + " " + label
            input_ids = self.tokenizer.encode(full_text, return_tensors="pt").to(self.model.device)
            prompt_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)
            prompt_len = prompt_ids.shape[1]
            with torch.no_grad():
                outputs = self.model(input_ids)
                logits = outputs.logits
            label_logits = logits[0, prompt_len-1 : -1, :] 
            label_ids = input_ids[0, prompt_len:]
            metadata[label] = {
                'input_ids': input_ids,
                'logits': logits,
                'label_ids': label_ids
            }
            loss = F.cross_entropy(label_logits, label_ids, reduction='sum')
            scores.append(-loss.item())
        return scores, metadata

In [24]:
lotan_v0 = LotanHFAgentV0(model, tokenizer)
lotan_v0_scores, lotan_v0_metadata = lotan_v0.query(q1, labels=['Option 1', 'Option 2'])
lotan_v0_scores

[-12.453125, -12.484375]

In [26]:
print(tokenizer.decode(lotan_v0_metadata['Option 1']['input_ids'][0]))

<|im_start|>system
You are a helpful assistant. Answer shortly with only your choice with no explanation.

<|im_end|>
<|im_start|>user
You have two options:
Option 1 - Green
Option 2 - Purple
Which do you prefer?
Answer: <|im_end|>
<|im_start|>assistant
 Option 1


In [27]:
lotan_v0_metadata['Option 1']

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
           21806,  19620,    448,   1172,    697,   5754,    448,    902,  16148,
             382, 151645,    198, 151644,    872,    198,   2610,    614,   1378,
            2606,    510,   5341,    220,     16,    481,   7840,    198,   5341,
             220,     17,    481,  40389,    198,  23085,    653,    498,  10702,
            5267,  16141,     25,    220, 151645,    198, 151644,  77091,    198,
            6959,    220,     16]]),
 'logits': tensor([[[ 3.8613,  7.9492,  3.2930,  ..., -1.7100, -1.7109, -1.7100],
          [ 5.5898, 11.1719,  7.4258,  ...,  0.1702,  0.1702,  0.1702],
          [ 7.4219, 10.5469, 13.2344,  ...,  0.3496,  0.3496,  0.3494],
          ...,
          [ 8.7344,  6.1211,  6.1836,  ..., -3.6484, -3.6484, -3.6484],
          [ 5.0664, -1.2500,  0.6655,  ..., -4.3438, -4.3438, -4.3438],
          [ 8.0938,  5.7422,  0.2139,  ..., -3.3555, -3.3555, -3.3555]]],

In [43]:
tokenizer.decode(lotan_v0_metadata['Option 1']['label_ids'])

' Option 1'

This space is the problem!

# Itay v0

In [ ]:
class ItayHFAgentV0(HFAgent):
    """ The original model from Itay's paper """
    def get_chat_format_one_side(self, text, role):
        return {
            "role": role,
            "content": text,
        }

    def convert_to_chat_format(self, text, few_shots_texts=None):
        """
        Uses the apply_chat_template function in the tokenizer of the predictor to convert the text to chat format
        """
        SYSTEM_MESSAGE = "You are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n"
        # if there are no few shots, just add the system message to the text
        if few_shots_texts is None:
            messages = [
                self.get_chat_format_one_side(SYSTEM_MESSAGE + text, "user"),
            ]
        # if there are few shots, add the system message to the first shot

        else:
            # the first shot needs the system message before the text
            messages = [
                self.get_chat_format_one_side(
                    SYSTEM_MESSAGE + few_shots_texts[0]["question"], "user"
                ),
                self.get_chat_format_one_side(
                    few_shots_texts[0]["answer"], "assistant"
                ),
            ]

            # after that, add all the other shot in the user and assistent format
            for shot in few_shots_texts[1:]:
                messages.extend(
                    [
                        self.get_chat_format_one_side(shot["question"], "user"),
                        self.get_chat_format_one_side(shot["answer"], "assistant"),
                    ]
                )

            # finaly, add the actual text to the user
            messages.append(self.get_chat_format_one_side(text, "user"))

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,  # , return_tensors="pt"
        )

        return prompt
    
    def query(self, prompt, labels):
        prompt = [self.convert_to_chat_format(p) for p in prompt]
        # concat labels to the corrposnded input text
        input_with_answers = [i + label for label in labels for i in prompt]
        # get labels tokens ids
        labels_tokens = self.tokenizer(labels, add_special_tokens=False)["input_ids"]
        # get the last token id of each label
        labels_tokens = [label[-1] for label in labels_tokens]
        # Ensure pad token exists before padding
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        # Get encodings for each input text using direct call instead of batch_encode_plus
        input_enc = self.tokenizer(
            input_with_answers,
            return_tensors="pt",
            padding="longest",
        )
        for k, v in input_enc.items():
            input_enc[k] = v.to(self.model.device)

        # Get model output logits
        model_output = self.model(**input_enc)

        # Compute the log probabilities associated with each of the labels
        labels_log_probs = F.log_softmax(model_output.logits, dim=-1)

        # Get the ids of the token before the last token before padding (to see the probablity of the last token given the one before the last token)
        before_padding_ids = (
            input_enc["input_ids"].ne(self.tokenizer.pad_token_id).sum(-1) - 2
        )

        # Collect labels scores from the -2 token in labels_log_probs (the one that predict the last token)
        # and collect for each line the id in labels_tokens
        labels_scores = labels_log_probs[:, before_padding_ids, labels_tokens]

        # Need just the diagonal of the matrix, as it the prob of the label for each line
        labels_scores = torch.diag(labels_scores)

        metadata = {
            'input_ids': input_enc.input_ids,
            'logits': model_output.logits,
        }

        return labels_scores, metadata


In [36]:
itay_v0 = ItayHFAgentV0(model, tokenizer)
itay_v0_scores, itay_v0_metadata = itay_v0.query([q1], labels=['Option 1','Option 2'])
itay_v0_scores

tensor([-0.8525, -0.5557], dtype=torch.float16,
       grad_fn=<DiagonalBackward0_copy>)

# Lotan v1

In [49]:
class LotanHFAgentV1(HFAgent):
    def query(self, prompt: str, labels: List[str]) -> List[float]:
        prompt = self._convert_to_chat_template(prompt)
        scores = []
        metadata = {}
        prompt_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)

        for label in labels:
            label_ids = self.tokenizer.encode(label, return_tensors="pt").to(self.model.device)
            input_ids = torch.cat([prompt_ids, label_ids], dim=1).to(self.model.device)
            prompt_len = prompt_ids.shape[1]
            with torch.no_grad():
                outputs = self.model(input_ids)
                logits = outputs.logits
            label_logits = logits[0, prompt_len-1 : -1, :] 
            label_ids = input_ids[0, prompt_len:]
            metadata[label] = {
                'input_ids': input_ids,
                'logits': logits,
                'label_ids': label_ids
            }
            loss = F.cross_entropy(label_logits, label_ids, reduction='sum')
            scores.append(-loss.item())
        return scores, metadata

In [50]:
lotan_v1 = LotanHFAgentV1(model, tokenizer)
lotan_v1_scores, lotan_v1_metadata = lotan_v1.query(q1, labels=['Option 1', 'Option 2'])
lotan_v1_scores

[-2.578125, -2.40625]

In [51]:
tokenizer.decode(lotan_v1_metadata['Option 1']['label_ids'])

'Option 1'

lotan_v1 calculates:
`Score = log P("Option" | prompt) + log P(" 1" | prompt + "Option")`

while itay_v0 calculates:
`Score = log P(" 1" | prompt + "Option")`

This adds a fixed "bias" for lotan_v1 equal `log P("Option" | prompt)`

In [53]:
tokenizer.decode(lotan_v1_metadata['Option 1']['input_ids'][0])

'<|im_start|>system\nYou are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n<|im_end|>\n<|im_start|>user\nYou have two options:\nOption 1 - Green\nOption 2 - Purple\nWhich do you prefer?\nAnswer: <|im_end|>\n<|im_start|>assistant\nOption 1'

# Itay v1

In [3]:
class ItayHFAgentV1(HFAgent):
    """
    - Fixed system prompt issue.
    - Used simpler convert_to_chat_format
    - covnerting prompts in the query function.
    """
    def query(self, prompts, labels):
        if not isinstance(prompts, list):
            prompts = [prompts]
        prompts = [self._convert_to_chat_template(p) for p in prompts]
        # concat labels to the corrposnded input text
        input_with_answers = [i + label for label in labels for i in prompts]
        # get labels tokens ids
        labels_tokens = self.tokenizer(labels, add_special_tokens=False)["input_ids"]
        # get the last token id of each label
        labels_tokens = [label[-1] for label in labels_tokens]
        # Ensure pad token exists before padding
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        # Get encodings for each input text using direct call instead of batch_encode_plus
        input_enc = self.tokenizer(
            input_with_answers,
            return_tensors="pt",
            padding="longest",
        )
        for k, v in input_enc.items():
            input_enc[k] = v.to(self.model.device)

        # Get model output logits
        model_output = self.model(**input_enc)

        # Compute the log probabilities associated with each of the labels
        labels_log_probs = F.log_softmax(model_output.logits, dim=-1)

        # Get the ids of the token before the last token before padding (to see the probablity of the last token given the one before the last token)
        before_padding_ids = (
            input_enc["input_ids"].ne(self.tokenizer.pad_token_id).sum(-1) - 2
        )

        # Collect labels scores from the -2 token in labels_log_probs (the one that predict the last token)
        # and collect for each line the id in labels_tokens
        labels_scores = labels_log_probs[:, before_padding_ids, labels_tokens]

        # Need just the diagonal of the matrix, as it the prob of the label for each line
        labels_scores = torch.diag(labels_scores)

        metadata = {
            'input_ids': input_enc.input_ids,
            'logits': model_output.logits,
        }

        return labels_scores, metadata


In [4]:
itay_v1 = ItayHFAgentV1(model, tokenizer)
itay_v1_scores, itay_v1_metadata = itay_v1.query([q1], labels=['Option 1', 'Option 2'])
itay_v1_scores

tensor([-0.7827, -0.6108], dtype=torch.float16,
       grad_fn=<DiagonalBackward0_copy>)

In [8]:
print(tokenizer.decode(itay_v1_metadata['input_ids'][0]))

<|im_start|>system
You are a helpful assistant. Answer shortly with only your choice with no explanation.

<|im_end|>
<|im_start|>user
You have two options:
Option 1 - Green
Option 2 - Purple
Which do you prefer?
Answer: <|im_end|>
<|im_start|>assistant
Option 1


In [ ]:
q2_template = "Which do you prefer: {A} or {B}?"
q2 = q2_template.format(A=options[0], B=options[1])

In [ ]:
itay_v1_scores_2, itay_v1_metadata_2 = itay_v1.query([q1], labels=options)
itay_v1_scores_2